# Phase 3B: Feature Engineering

**Objective:** Transform the cleaned "Golden Dataset" into a numeric feature matrix optimized for Machine Learning.

This phase includes structural data preparation, categorical encoding, numerical transformations, derived feature creation, and finalizing the dataset for modeling.

## 1. Imports, DB Connection & Data Loading
Re-establishing the database connection to load the raw data and applying the same credibility filters from the EDA phase to ensure consistency.
```

In [23]:
import os
import ast
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

load_dotenv('../.env')

POSTGRES_HOST = os.getenv("POSTGRES_HOST", os.getenv("DB_HOST", "localhost"))
POSTGRES_PORT = os.getenv("POSTGRES_PORT", os.getenv("DB_PORT", "5432"))
POSTGRES_DB   = os.getenv("POSTGRES_DB",   os.getenv("DB_NAME", "cinema_intelligence"))
POSTGRES_USER = os.getenv("POSTGRES_USER", os.getenv("DB_USER", "cinema_user"))
POSTGRES_PASS = os.getenv("POSTGRES_PASSWORD", os.getenv("DB_PASS", "cinema_password"))

PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

def get_conn():
    return psycopg2.connect(
        host=POSTGRES_HOST, port=POSTGRES_PORT,
        dbname=POSTGRES_DB, user=POSTGRES_USER, password=POSTGRES_PASS
    )

query = """
    SELECT
        m.movie_id,
        m.external_id,
        m.title,
        m.release_date,
        EXTRACT(YEAR  FROM m.release_date)::INT AS release_year,
        m.runtime_minutes,
        m.market_type,
        m.rating,
        m.vote_count,
        l.language_name,
        ARRAY_AGG(g.genre_name) FILTER (WHERE g.genre_name IS NOT NULL) AS genres
    FROM movies m
    LEFT JOIN languages    l  ON m.language_id  = l.language_id
    LEFT JOIN movie_genres mg ON m.movie_id      = mg.movie_id
    LEFT JOIN genres       g  ON mg.genre_id     = g.genre_id
    WHERE m.release_date IS NOT NULL
    GROUP BY m.movie_id, m.external_id, m.title, m.release_date,
             m.runtime_minutes, m.market_type, m.rating, m.vote_count, l.language_name
"""

conn = get_conn()
df = pd.read_sql(query, conn)
conn.close()

df["release_date"] = pd.to_datetime(df["release_date"])

df_clean = df[
    (df["runtime_minutes"].isna() | (df["runtime_minutes"] <= 400)) &
    (df["vote_count"].isna() | (df["vote_count"] >= 50))
].copy()

print(f"Raw shape        : {df.shape}")
print(f"Clean shape      : {df_clean.shape}")
print(f"Rows removed     : {df.shape[0] - df_clean.shape[0]}")
print(f"Years covered    : {df_clean['release_year'].min()} – {df_clean['release_year'].max()}")
print(f"\nMarket split (clean):")
print(df_clean["market_type"].value_counts().to_string())
print(f"\nDtypes:")
print(df_clean.dtypes.to_string())


C:\Users\periy\AppData\Local\Temp\ipykernel_28820\1488075444.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Raw shape        : (59905, 11)
Clean shape      : (45882, 11)
Rows removed     : 14023
Years covered    : 2021 – 2025

Market split (clean):
market_type
Hollywood    36233
Indian        9649

Dtypes:
movie_id                   int64
external_id                  str
title                        str
release_date       datetime64[s]
release_year               int64
runtime_minutes          float64
market_type                  str
rating                   float64
vote_count               float64
language_name                str
genres                    object


---
## 2. Temporal Features: `release_year` Only

**Design Decision — Sub-Year Features Excluded**

The source dataset (IMDb basics) stores `release_date` at year-only precision — every entry defaults to `January 1st` of its release year.

As a result, the following features would carry **zero variance or false signals** and are deliberately **not engineered**:
- `release_month` — always January (1)
- `release_quarter` — always Q1
- `is_weekend_release` — Jan 1 day-of-week varies by year, not by actual release intent

**`release_year` is retained as the sole temporal feature** (integer, 2021–2025). It captures linear industry trends across the dataset window. In the Modeling phase, it can optionally be one-hot encoded to isolate year-specific market conditions (e.g., the 2023 WGA/SAG-AFTRA industry strikes).
```


---
## 3. Categorical Encoding (Text to Math)

### Multi-Label Genre Encoding
Transforming the `genres` list into individual binary columns using `MultiLabelBinarizer`. This creates a sparse matrix where each unique genre becomes a feature.
```

In [ ]:
mlb = MultiLabelBinarizer()

genres_filled = df_clean["genres"].apply(
    lambda x: x if isinstance(x, list) else []
)

genre_matrix = mlb.fit_transform(genres_filled)
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=df_clean.index)

print(f"Total unique genres : {len(mlb.classes_)}")
print(f"Genre columns       : {list(mlb.classes_)}")
print(f"Genre matrix shape  : {genre_df.shape}")


Total unique genres : 26
Genre columns       : ['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']
Genre matrix shape  : (45882, 26)


---
### Language Target Encoding
Replacing categorical language names with a "Quality Signal" by mapping each language to its global average rating within the dataset.
```

In [ ]:
global_mean = df_clean["rating"].mean()

lang_avg = df_clean.groupby("language_name")["rating"].mean().round(4)

df_clean["language_quality_score"] = (
    df_clean["language_name"]
    .map(lang_avg)
    .fillna(global_mean)
)

print(f"Global average rating : {global_mean:.4f}")
print(f"\nLanguage → Avg Rating mapping:")
print(lang_avg.sort_values(ascending=False).to_string())


Global average rating : 5.9244

Language → Avg Rating mapping:
language_name
Gujarati     7.4750
Marathi      7.4667
Urdu         7.4400
Kannada      6.5000
Tamil        6.3345
Telugu       6.1793
Punjabi      6.1000
Malayalam    6.0295
Bengali      6.0056
Hindi        5.9214
English      5.9206


---
### Market Encoding
Applying One-Hot Encoding to the `market_type` to separate Hollywood and Indian cinema flags into distinct binary columns.

In [ ]:
market_dummies = pd.get_dummies(df_clean["market_type"], prefix="market")

print(f"Market columns created : {list(market_dummies.columns)}")
print(f"\nValue counts:")
print(market_dummies.sum().to_string())


Market columns created : ['market_Hollywood', 'market_Indian']

Value counts:
market_Hollywood    36233
market_Indian        9649


---
## 4. Numerical Transformation & Scaling

To ensure numerical inputs occupy the same mathematical space and prevent high-magnitude features from dominating:
1. **Missing Data Imputation:** Filling missing `runtime_minutes` with the median.
2. **Skewness Correction:** Applying a Log Transformation ($\log(x+1)$) to `vote_count` to handle its right-skewed distribution.
3. **Feature Scaling:** Using `StandardScaler` on continuous features. **Crucially**, the scaler is fitted *only* on the training data (years < 2025) to prevent data leakage.

In [ ]:
runtime_median = df_clean["runtime_minutes"].median()
df_clean["runtime_minutes_filled"] = df_clean["runtime_minutes"].fillna(runtime_median)

df_clean["vote_count_log"] = np.log1p(df_clean["vote_count"])

train_mask = df_clean["release_year"] < 2025
scaler = StandardScaler()
scaler.fit(df_clean.loc[train_mask, ["runtime_minutes_filled", "vote_count_log"]])

scaled = scaler.transform(df_clean[["runtime_minutes_filled", "vote_count_log"]])
df_clean["runtime_minutes_scaled"] = scaled[:, 0]
df_clean["vote_count_log_scaled"]  = scaled[:, 1]

print(f"Runtime median used for fill : {runtime_median:.1f} min")
print(f"\nScaler fitted on train rows  : {train_mask.sum():,}")
print(f"\nruntime_minutes_scaled — mean: {df_clean.loc[train_mask,'runtime_minutes_scaled'].mean():.4f}, std: {df_clean.loc[train_mask,'runtime_minutes_scaled'].std():.4f}")
print(f"vote_count_log_scaled  — mean: {df_clean.loc[train_mask,'vote_count_log_scaled'].mean():.4f}, std: {df_clean.loc[train_mask,'vote_count_log_scaled'].std():.4f}")


Runtime median used for fill : 95.0 min

Scaler fitted on train rows  : 36,737

runtime_minutes_scaled — mean: 0.0000, std: 1.0000
vote_count_log_scaled  — mean: 0.0000, std: 1.0000


---
## 5. Derived Feature Creation

Creating new signals from existing data:
- `genre_density`: The count of genres assigned to a movie.
- `votes_per_minute`: Measuring audience engagement relative to runtime. Unrated movies (NaN) are filled with 0.

In [ ]:
df_clean["genre_density"] = genres_filled.apply(len)

df_clean["votes_per_minute"] = (
    df_clean["vote_count"] / df_clean["runtime_minutes_filled"]
)
df_clean["votes_per_minute"] = df_clean["votes_per_minute"].fillna(0)

print("genre_density stats:")
print(df_clean["genre_density"].describe().round(2).to_string())
print(f"\nvotes_per_minute stats:")
print(df_clean["votes_per_minute"].describe().round(2).to_string())
print(f"\nNaN in votes_per_minute : {df_clean['votes_per_minute'].isna().sum()}")


genre_density stats:
count    45882.00
mean         1.55
std          0.83
min          0.00
25%          1.00
50%          1.00
75%          2.00
max          3.00

votes_per_minute stats:
count    45882.00
mean        24.00
std        171.71
min          0.00
25%          0.00
50%          0.75
75%          4.33
max       6891.20

NaN in votes_per_minute : 0


---
## 6. Modeling Readiness

### Target Definition
Defining our target variables for the models:
- **Regression:** `rating` (continuous score)
- **Classification:** `is_hit` (binary flag where rating > 7.0)

Movies without a target rating are dropped at this stage.

In [ ]:
df_model = df_clean.dropna(subset=["rating"]).copy()

df_model["is_hit"] = (df_model["rating"] > 7.0).astype(int)

print(f"Rows before dropping NaN rating : {len(df_clean):,}")
print(f"Rows after  dropping NaN rating : {len(df_model):,}")
print(f"Rows dropped                    : {len(df_clean) - len(df_model):,}")
print(f"\nis_hit distribution:")
print(df_model["is_hit"].value_counts().to_string())
print(f"\nRating stats:")
print(df_model["rating"].describe().round(2).to_string())


Rows before dropping NaN rating : 45,882
Rows after  dropping NaN rating : 25,417
Rows dropped                    : 20,465

is_hit distribution:
is_hit
0    19802
1     5615

Rating stats:
count    25417.00
mean         5.92
std          1.46
min          1.00
25%          5.00
50%          6.00
75%          6.90
max          9.90


---
### Feature Matrix Assembly
Concatenating all engineered numerical features, market dummies, and genre matrices into the final Feature Matrix (`X`). Raw text and pre-transformed columns are dropped.

In [ ]:
genre_df_model = genre_df.loc[df_model.index]
market_dummies_model = market_dummies.loc[df_model.index]

X = pd.concat([
    df_model[["release_year", "language_quality_score",
              "runtime_minutes_scaled", "vote_count_log_scaled",
              "genre_density", "votes_per_minute"]],
    market_dummies_model,
    genre_df_model
], axis=1)

y_regression     = df_model["rating"].reset_index(drop=True)
y_classification = df_model["is_hit"].reset_index(drop=True)
X = X.reset_index(drop=True)

print(f"Feature matrix shape : {X.shape}")
print(f"\nColumns ({len(X.columns)}):")
print(list(X.columns))
print(f"\nNaN check (should all be 0):")
print(X.isnull().sum()[X.isnull().sum() > 0].to_string() or "No NaN found")
print(f"\nDtypes:")
print(X.dtypes.value_counts().to_string())


Feature matrix shape : (25417, 34)

Columns (34):
['release_year', 'language_quality_score', 'runtime_minutes_scaled', 'vote_count_log_scaled', 'genre_density', 'votes_per_minute', 'market_Hollywood', 'market_Indian', 'Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']

NaN check (should all be 0):
Series([], )

Dtypes:
int64      28
float64     4
bool        2


---
### Chronological Train/Test Split
Partitioning the data chronologically to evaluate the model on "future" unseen data:
- **Training Set:** 2021–2024
- **Test Set:** 2025

In [ ]:
X = X.astype({col: int for col in X.select_dtypes("bool").columns})

train_mask = df_model["release_year"].values < 2025
test_mask  = df_model["release_year"].values == 2025

X_train, X_test = X[train_mask], X[test_mask]
y_train_reg,  y_test_reg  = y_regression[train_mask],     y_regression[test_mask]
y_train_cls,  y_test_cls  = y_classification[train_mask], y_classification[test_mask]

print(f"Train rows : {len(X_train):,}  ({df_model['release_year'].value_counts().sort_index()[:-1].to_dict()})")
print(f"Test  rows : {len(X_test):,}   (2025 only)")
print(f"\nTrain is_hit → 0: {(y_train_cls==0).sum():,}  |  1: {(y_train_cls==1).sum():,}")
print(f"Test  is_hit → 0: {(y_test_cls==0).sum():,}    |  1: {(y_test_cls==1).sum():,}")
print(f"\nAll dtypes numeric: {all(X_train.dtypes != 'bool')}")


Train rows : 21,161  ({2021: 4813, 2022: 5407, 2023: 5597, 2024: 5344})
Test  rows : 4,256   (2025 only)

Train is_hit → 0: 16,671  |  1: 4,490
Test  is_hit → 0: 3,131    |  1: 1,125

All dtypes numeric: True


---
### Export Artifacts
Saving the final matrices as `.parquet` files for efficient reading in the modeling phase, along with the fitted scaler and binarizer `.pkl` files for future inference.

In [ ]:
import pickle

X_train.to_parquet(PROCESSED_PATH / "X_train.parquet", index=False)
X_test.to_parquet(PROCESSED_PATH / "X_test.parquet", index=False)

y_train_reg.to_frame("rating").to_parquet(PROCESSED_PATH / "y_train_regression.parquet", index=False)
y_test_reg.to_frame("rating").to_parquet(PROCESSED_PATH / "y_test_regression.parquet", index=False)

y_train_cls.to_frame("is_hit").to_parquet(PROCESSED_PATH / "y_train_classification.parquet", index=False)
y_test_cls.to_frame("is_hit").to_parquet(PROCESSED_PATH / "y_test_classification.parquet", index=False)

with open(PROCESSED_PATH / "standard_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(PROCESSED_PATH / "genre_mlb.pkl", "wb") as f:
    pickle.dump(mlb, f)

print("Files saved to:", PROCESSED_PATH.resolve())
for p in sorted(PROCESSED_PATH.glob("*")):
    print(f"  {p.name:<40} {p.stat().st_size / 1024:.1f} KB")


Files saved to: C:\Folders\Movie Data Analysis\data\processed
  genre_mlb.pkl                            0.5 KB
  standard_scaler.pkl                      0.6 KB
  target_list.parquet                      1822.5 KB
  tmdb_checkpoint.json                     0.0 KB
  X_test.parquet                           140.3 KB
  X_train.parquet                          731.7 KB
  y_test_classification.parquet            4.6 KB
  y_test_regression.parquet                9.8 KB
  y_train_classification.parquet           17.8 KB
  y_train_regression.parquet               45.9 KB
